# Stage 2 — Instruction Fine-Tuning (SFT)

**Domain:** Finance FAQ Assistant
**Starting point:** the Stage 1 adapter (`outputs/non_instruction_adapter`)
**Data:** `data/instruction_dataset.jsonl` (100+ instruction/response pairs)
**Goal:** teach the model to actually answer finance questions well, in a
helpful Q&A format, building on the domain adaptation from Stage 1.

> Run on Colab with a T4 GPU.


In [ ]:
%%capture
!pip install unsloth


In [ ]:
from huggingface_hub import login
from google.colab import userdata

# One-time setup: click the key icon in Colab's left sidebar, add a secret
# named HF_TOKEN with your Hugging Face write token, and toggle notebook
# access on. It then persists across every future Colab session automatically.
login(token=userdata.get("HF_TOKEN"))


In [ ]:
from google.colab import files
import os

for fname in ["data/instruction_dataset.jsonl"]:
    if not os.path.exists(fname):
        os.makedirs("data", exist_ok=True)
        print(f"Upload {os.path.basename(fname)}:")
        uploaded = files.upload()
        for f in uploaded:
            os.rename(f, f"data/{f}")

# If you also have the Stage 1 adapter folder, upload/clone it too so this
# notebook can continue from it instead of the raw base model:
#   outputs/non_instruction_adapter/


In [ ]:
from unsloth import FastLanguageModel
import torch, os

max_seq_length = 2048

HF_USERNAME = "Naveengangadhara"
STAGE1_REPO = f"{HF_USERNAME}/finance-qwen-stage1-adapter"
STAGE1_ADAPTER_LOCAL = "outputs/non_instruction_adapter"

# Prefer the local adapter if it's in this same runtime (fast), otherwise
# pull the Stage 1 adapter pushed to the Hub (survives runtime disconnects).
if os.path.isdir(STAGE1_ADAPTER_LOCAL):
    base_model_name = STAGE1_ADAPTER_LOCAL
else:
    base_model_name = STAGE1_REPO
print(f"Loading from: {base_model_name}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = base_model_name,
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)


In [ ]:
# If we loaded the raw base model (no Stage 1 adapter available), attach a
# fresh LoRA adapter now. If we loaded the Stage 1 adapter directly, Unsloth
# returns it already PEFT-wrapped, so skip re-wrapping.
from peft import PeftModel

if not isinstance(model, PeftModel):
    model = FastLanguageModel.get_peft_model(
        model,
        r = 16,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                           "gate_proj", "up_proj", "down_proj"],
        lora_alpha = 16,
        lora_dropout = 0,
        bias = "none",
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
    )


## Format the instruction dataset

Each line of `data/instruction_dataset.jsonl` looks like:

```json
{"instruction": "How can I apply for sick leave?", "response": "..."}
```

We wrap these into a simple Alpaca-style prompt template so the model learns
a consistent instruction → response pattern.


In [ ]:
import json
from datasets import Dataset

PROMPT_TEMPLATE = """Below is a question about personal finance. Write a clear, accurate, and helpful response.

### Question:
{instruction}

### Response:
{response}"""

EOS_TOKEN = tokenizer.eos_token

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

raw_rows = load_jsonl("data/instruction_dataset.jsonl")
print(f"Loaded {len(raw_rows)} instruction/response pairs")

def format_example(example):
    text = PROMPT_TEMPLATE.format(
        instruction=example["instruction"], response=example["response"]
    ) + EOS_TOKEN
    return {"text": text}

dataset = Dataset.from_list(raw_rows).map(format_example)


In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_length = max_seq_length,
        packing = False,
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs/instruction_sft",
        report_to = "none",
    ),
)


In [ ]:
trainer_stats = trainer.train()


In [ ]:
STAGE2_REPO = f"{HF_USERNAME}/finance-qwen-sft-adapter"

model.save_pretrained("outputs/sft_adapter")
tokenizer.save_pretrained("outputs/sft_adapter")
print("Saved Stage 2 (SFT) adapter to outputs/sft_adapter")

# Push to Hugging Face Hub so Stage 3 (DPO) can load it even in a fresh runtime
model.push_to_hub(STAGE2_REPO)
tokenizer.push_to_hub(STAGE2_REPO)
print(f"Pushed Stage 2 adapter to https://huggingface.co/{STAGE2_REPO}")


## Inference after instruction fine-tuning

Ask one of the 10 evaluation questions and compare qualitatively against the
base model output you recorded in `reports/base_model_evaluation.md`.


In [ ]:
FastLanguageModel.for_inference(model)

def ask(question, max_new_tokens=200):
    prompt = PROMPT_TEMPLATE.format(instruction=question, response="")
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, use_cache=True)
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded.split("### Response:")[-1].strip()

print(ask("How can I start building an emergency fund?"))
